# 01 - Data Setup and Audit

This notebook prepares the QCRI/HumAID-all crisis-message splits for the CrisisText project. It keeps the data work local and reproducible: no Colab mounts, no Google Drive paths, and no model training.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import sys
import unicodedata

import pandas as pd
from datasets import load_dataset

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.preprocessing import preprocess_text

pd.set_option("display.max_colwidth", 160)

## Dataset Download

The authoritative dataset is `QCRI/HumAID-all` on Hugging Face. The expected splits are train, validation, and test.

In [ ]:
DATASET_NAME = "QCRI/HumAID-all"
dataset = load_dataset(DATASET_NAME)
dataset

In [ ]:
split_summary = pd.DataFrame(
    {
        "split": list(dataset.keys()),
        "rows": [len(dataset[split]) for split in dataset.keys()],
        "columns": [dataset[split].column_names for split in dataset.keys()],
    }
)
split_summary

## DataFrame Conversion

The public dataset contains message text and a humanitarian class label. The local parquet files are optional regeneration artifacts and are not committed to Git.

In [ ]:
train_df = dataset["train"].to_pandas()
validation_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

for name, frame in {"train": train_df, "validation": validation_df, "test": test_df}.items():
    print(name, frame.shape, list(frame.columns))

In [ ]:
def audit_split(name: str, frame: pd.DataFrame) -> dict[str, object]:
    return {
        "split": name,
        "rows": len(frame),
        "columns": list(frame.columns),
        "missing_tweet_text": int(frame["tweet_text"].isna().sum()),
        "missing_class_label": int(frame["class_label"].isna().sum()),
        "duplicate_rows": int(frame.duplicated().sum()),
        "unique_labels": int(frame["class_label"].nunique()),
    }

audit_df = pd.DataFrame(
    audit_split(name, frame)
    for name, frame in {"train": train_df, "validation": validation_df, "test": test_df}.items()
)
audit_df

## Label Distribution and Imbalance

The model-selection notebooks use Macro-F1 because operational minority classes matter and the class distribution is imbalanced.

In [ ]:
label_counts = (
    train_df["class_label"]
    .value_counts()
    .rename_axis("class_label")
    .reset_index(name="train_count")
)
label_counts["train_percentage"] = label_counts["train_count"] / len(train_df) * 100
imbalance_ratio = label_counts["train_count"].max() / label_counts["train_count"].min()
print(f"Train imbalance ratio: {imbalance_ratio:.2f}:1")
label_counts

## Message Lengths

Character and word counts help identify truncated, unusually short, or unusually long messages.

In [ ]:
train_audit_df = train_df.copy()
train_audit_df["char_count"] = train_audit_df["tweet_text"].str.len()
train_audit_df["word_count"] = train_audit_df["tweet_text"].str.split().str.len()

length_summary = train_audit_df[["char_count", "word_count"]].describe().round(2)
length_summary

In [ ]:
short_examples = train_audit_df.nsmallest(10, "char_count")[["tweet_text", "class_label", "char_count", "word_count"]]
long_examples = train_audit_df.nlargest(10, "char_count")[["tweet_text", "class_label", "char_count", "word_count"]]
short_examples, long_examples

## Social-Media Text Signals

The dataset contains URLs, mentions, hashtags, retweets, punctuation, and occasional unusual Unicode sequences. These are audited before any preprocessing choice is made.

In [ ]:
signal_patterns = {
    "has_url": r"https?://\S+|www\.\S+",
    "has_mention": r"@\w+",
    "has_hashtag": r"#\w+",
    "is_retweet": r"^RT\s+",
    "has_repeated_punctuation": r"([!?.,])\1{1,}",
}

for column, pattern in signal_patterns.items():
    train_audit_df[column] = train_audit_df["tweet_text"].str.contains(pattern, regex=True, na=False)

signal_summary = (
    train_audit_df[list(signal_patterns)]
    .mean()
    .mul(100)
    .round(2)
    .rename("percentage")
    .reset_index(names="signal")
)
signal_summary

In [ ]:
train_audit_df["mention_count"] = train_audit_df["tweet_text"].str.count(r"@\w+")
train_audit_df.nlargest(10, "mention_count")[["tweet_text", "class_label", "mention_count", "char_count"]]

In [ ]:
def has_suspicious_unicode(text: str) -> bool:
    return any(
        unicodedata.category(character).startswith("C") and character not in "\n\t"
        for character in text
    )

train_audit_df["has_suspicious_unicode"] = train_audit_df["tweet_text"].apply(has_suspicious_unicode)
train_audit_df["has_mojibake_hint"] = train_audit_df["tweet_text"].str.contains(r"[??????]", regex=True, na=False)

unicode_summary = train_audit_df[["has_suspicious_unicode", "has_mojibake_hint"]].mean().mul(100).round(3)
unicode_examples = train_audit_df.loc[
    train_audit_df["has_suspicious_unicode"] | train_audit_df["has_mojibake_hint"],
    ["tweet_text", "class_label"],
].head(10)
unicode_summary, unicode_examples

## Minimal Preprocessing

The project keeps raw text for the final selected model, but the original experiment journey also compares a minimal preprocessing variant. The function below is imported from `src.preprocessing`.

In [ ]:
example_messages = pd.Series(
    [
        "  HELP!!!\n\nWe need   WATER  ",
        "RT @RachelAndJun: Please help @UNICEF",
        "Donate at https://example.com/help #FloodHelp",
    ],
    name="raw_text",
)

preprocessing_examples = pd.DataFrame(
    {
        "raw_text": example_messages,
        "text_minimal": example_messages.apply(preprocess_text),
    }
)
preprocessing_examples

In [ ]:
prepared_splits = {
    "train": train_df.copy(),
    "validation": validation_df.copy(),
    "test": test_df.copy(),
}

for frame in prepared_splits.values():
    frame["text_minimal"] = frame["tweet_text"].apply(preprocess_text)

prepared_summary = pd.DataFrame(
    {
        "split": name,
        "rows": len(frame),
        "empty_minimal_text": int(frame["text_minimal"].str.strip().eq("").sum()),
        "changed_by_preprocessing": int(frame["tweet_text"].ne(frame["text_minimal"]).sum()),
    }
    for name, frame in prepared_splits.items()
)
prepared_summary

## Optional Local Save

These files are useful for local reproduction and are intentionally ignored by Git. Keep `SAVE_LOCAL_PARQUET` false when only auditing interactively.

In [ ]:
SAVE_LOCAL_PARQUET = False

if SAVE_LOCAL_PARQUET:
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_df.to_parquet(RAW_DATA_DIR / "humaid_train.parquet", index=False)
    validation_df.to_parquet(RAW_DATA_DIR / "humaid_validation.parquet", index=False)
    test_df.to_parquet(RAW_DATA_DIR / "humaid_test.parquet", index=False)

    prepared_splits["train"].to_parquet(PROCESSED_DATA_DIR / "humaid_train_minimal.parquet", index=False)
    prepared_splits["validation"].to_parquet(PROCESSED_DATA_DIR / "humaid_validation_minimal.parquet", index=False)
    prepared_splits["test"].to_parquet(PROCESSED_DATA_DIR / "humaid_test_minimal.parquet", index=False)

    print("Saved local parquet files under", RAW_DATA_DIR.parent)
else:
    print("Local parquet export skipped.")

## Summary

The train split contains ten labels with substantial imbalance, social-media artifacts are common, and minimal preprocessing is available for controlled ablations. Final model selection is handled in `02_model_training.ipynb`; the test set is not evaluated here.